# dsfb-robotics — paper §10 reproduction notebook

**What this notebook does.** Builds the `paper-lock` binary from source,
runs it in fixture mode for all ten datasets, and renders the companion
paper's §10 figures inline.

**Honesty disclosure.** This notebook runs against the **in-crate
micro-fixtures**, not the real datasets. The paper's empirical
numbers in §10 come from `paper-lock` runs against the full public
corpora (CWRU, IMS, C-MAPSS, KUKA LWR, FEMTO-ST PRONOSTIA, Franka
Panda Gaz 2019, DLR Rollin' Justin / LWR-III, UR10 Kufieta 2014,
MIT Cheetah 3, IIT iCub push-recovery). To reproduce those numbers,
obtain the corpora under each dataset's upstream licence / data-use
agreement (see each `docs/<slug>_oracle_protocol.md`) and run
`paper-lock <slug>` without the `--fixture` flag.

**Reproducibility.** Three consecutive invocations of each
`paper-lock --fixture` cell produce byte-identical JSON output.
This is the same tolerance gate enforced by
`tests/paper_lock_binary.rs::fixture_output_is_bit_exact_across_repeat_invocations`.

**Positioning.** DSFB does **not** compete with existing robotics
health monitoring, inverse-dynamics identification, whole-body
balance control, or prognostics methods. It reads the residuals
those methods already produce and usually discard, and structures
them into a human-readable grammar. Removing DSFB leaves the
upstream stack unchanged.

## 1. Prepare the environment

Clones the repository (if running on Colab) and installs the pinned
Rust toolchain. Skip this cell if you already have the crate checked
out and `paper-lock` on PATH.

In [ ]:
import os
import pathlib
import shutil
import subprocess
import sys

CRATE_DIR = pathlib.Path("dsfb-robotics")

if not CRATE_DIR.exists():
    # Colab path: clone the monorepo and cd into the crate.
    subprocess.run(
        ["git", "clone", "--depth=1", "https://github.com/infinityabundance/dsfb.git"],
        check=True,
    )
    CRATE_DIR = pathlib.Path("dsfb/crates/dsfb-robotics").resolve()
else:
    CRATE_DIR = CRATE_DIR.resolve()

if shutil.which("cargo") is None:
    # Install rustup via the canonical one-liner; pinned toolchain then
    # auto-resolves from the crate's rust-toolchain.toml.
    subprocess.run(
        "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain none",
        shell=True,
        check=True,
    )
    cargo_bin = pathlib.Path.home() / ".cargo" / "bin"
    os.environ["PATH"] = f"{cargo_bin}:{os.environ.get('PATH', '')}"

print("Cargo:", shutil.which("cargo"))
print("Crate:", CRATE_DIR)

## 2. Build the `paper-lock` binary

This is the only compile step in the notebook. Compilation with the
`std,paper_lock` features produces a small release binary — takes
≈ 3–4 minutes on free-tier Colab the first time; cached after that.

In [ ]:
result = subprocess.run(
    [
        "cargo", "build", "--release",
        "--bin", "paper-lock",
        "--features", "std,paper_lock",
    ],
    cwd=str(CRATE_DIR),
    check=False,
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print("STDOUT:", result.stdout)
    print("STDERR:", result.stderr)
    raise RuntimeError(f"cargo build failed with exit {result.returncode}")

BIN = CRATE_DIR / "target" / "release" / "paper-lock"
assert BIN.is_file(), f"binary not found: {BIN}"
print("paper-lock:", BIN)

## 3. Verify bit-exact determinism

Runs `paper-lock kuka_lwr --fixture --emit-episodes` three times and
confirms the JSON output is byte-identical — the reproducibility
tolerance gate.

In [ ]:
def capture(slug: str, emit_episodes: bool = False) -> str:
    args = [str(BIN), slug, "--fixture"]
    if emit_episodes:
        args.append("--emit-episodes")
    r = subprocess.run(args, check=True, capture_output=True, text=True)
    return r.stdout

a = capture("kuka_lwr", emit_episodes=True)
b = capture("kuka_lwr", emit_episodes=True)
c = capture("kuka_lwr", emit_episodes=True)
assert a == b == c, "paper-lock output drifted across invocations"
print("[determinism] kuka_lwr fixture runs are bit-identical ✓")

## 4. Run all ten datasets and collect reports

In [ ]:
import json

SLUGS = [
    "cwru", "ims", "cmapss", "kuka_lwr", "femto_st",
    "panda_gaz", "dlr_justin", "ur10_kufieta",
    "cheetah3", "icub_pushrecovery",
]
reports = {}
for slug in SLUGS:
    reports[slug] = json.loads(capture(slug, emit_episodes=True))
    agg = reports[slug]["aggregate"]
    print(
        f"  {slug:<20s} total={agg['total_samples']:>3d} "
        f"adm={agg['admissible']:>2d} bnd={agg['boundary']:>2d} "
        f"vio={agg['violation']:>2d} compression={agg['compression_ratio']:.3f}"
    )

## 5. Render figures inline

Re-uses the rendering functions from `scripts/figures_real.py` so
the notebook and the host-side figure script are guaranteed to
produce identical output.

In [ ]:
sys.path.insert(0, str(CRATE_DIR))
from scripts.figures_real import (
    Report,
    render_grammar_timeline,
    render_residual_on_envelope,
    render_compression_histogram,
)

OUT = CRATE_DIR / "dsfb-robotics-output" / "figs"
OUT.mkdir(parents=True, exist_ok=True)

parsed = [Report.load(r) for r in reports.values()]
written = []
for rpt in parsed:
    base = OUT / rpt.dataset
    written += render_grammar_timeline(rpt, base)
    written += render_residual_on_envelope(rpt, base)
written += render_compression_histogram(parsed, OUT / "_all")
print(f"wrote {len(written)} figure files to {OUT}")

## 6. Display representative figures inline

In [ ]:
from IPython.display import Image, display

for slug in ["kuka_lwr", "cheetah3", "icub_pushrecovery"]:
    display(Image(str(OUT / f"{slug}_grammar_timeline.png")))
display(Image(str(OUT / "_all_compression_histogram.png")))

## 7. Operator-facing flags: `--explain` (post-commit narrative)

The production binary supports `--explain`, which augments the
JSON output with an `explain[]` array. Each entry is a
post-commit narrative for one episode, naming the structural
condition that fired the FSM transition. The cell below runs
`paper-lock panda_gaz --fixture --explain` and pretty-prints
the first few entries of the resulting array.

Read the narrative as a triage prompt, not as a fault
explanation; see `docs/DEPLOYMENT_ANTIPATTERNS.md` anti-pattern 4
for the operator-facing reading rule.


In [ ]:
result = subprocess.run(
    [str(BIN), "panda_gaz", "--fixture", "--explain"],
    cwd=str(CRATE_DIR), capture_output=True, text=True, check=True,
)
doc = json.loads(result.stdout)
explain = doc.get("explain", [])
print(f"explain[] entries: {len(explain)}")
for entry in explain[:5]:
    print(json.dumps(entry, indent=2))
print("  ...")
print(f"(showing first 5 of {len(explain)} narratives;"
      f" full array lives in the JSON output above)")


## 8. Next steps

- To reproduce the paper's real-data numbers, obtain each dataset
  under its upstream licence and re-run `paper-lock <slug>` without
  the `--fixture` flag.
- To extend the figure set, edit `scripts/figures_real.py` — the
  notebook pulls its rendering functions from that file, so any
  additions appear automatically on the next Run-All.
- For deeper audit artefacts (Miri, Kani, cargo-fuzz, loom), see
  `audit/` and `REPRODUCE.md`.